# Survival & anomaly-detection metrics

_Metrics & Evaluation — measuring models, data & statistics_

**Two hard cases: scoring WHEN an event happens when some events have not happened yet, and finding rare events without being fooled by accuracy.**

Both families exist because a normal metric would either throw away information or be gamed by the majority class.

       Survival. Suppose you study 100 patients for 5 years and ask "when does each relapse?". By year 5, many have not relapsed. You do not know their true relapse time — only that it is longer than 5 years. That is right-censoring: you know a lower bound on the event time, not the event time itself. A censored row is real data ("survived at least this long"), and the right metrics use it.

---

This notebook is a practice scaffold for the **Survival & anomaly-detection metrics** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-survival / lifelines

### Step 1 — Fit a Cox model and read its risk scoresSurvival data has two parts per patient: whether the event was observed, and the time. We load the GBSG2 breast-cancer data, one-hot encode it, and fit a **Cox proportional-hazards** model.The model outputs a **risk score** per patient — higher means expected to fail sooner. We also pull out the `event` flag (True = event observed, False = right-censored) and the `time`, which the censoring-aware metrics need.

In [ ]:
# SURVIVAL: scikit-survival provides the censoring-aware metrics.
from sksurv.datasets import load_gbsg2
from sksurv.preprocessing import OneHotEncoder
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import (
    concordance_index_censored,   # Harrell's C
    concordance_index_ipcw,       # Uno's C (IPCW-corrected)
    cumulative_dynamic_auc,       # time-dependent AUC
    integrated_brier_score,       # IBS
)

X, y = load_gbsg2()                       # y is a structured array: (event_bool, time)
X = OneHotEncoder().fit_transform(X)
model = CoxPHSurvivalAnalysis().fit(X, y)

# Risk scores: higher = expected to fail sooner.
risk = model.predict(X)
event = y["cens"].astype(bool)            # True = event observed, False = censored
time = y["time"]

### Step 2 — Censoring-aware ranking and calibration metrics**Harrell's C-index** measures how often the model ranks the earlier-failing patient as higher-risk (0.5 = random, 1.0 = perfect). **Uno's C** is an IPCW-corrected version that down-weights the bias from censoring.Going beyond ranking, **time-dependent AUC** and the **Integrated Brier Score (IBS)** judge the predicted *survival curves* over a grid of times — IBS is a calibration score where lower is better.

In [ ]:
# Harrell's C-index (returns C first, then concordant/discordant/tied counts).
c_harrell = concordance_index_censored(event, time, risk)[0]

# Uno's C-index needs a training survival array to fit the censoring weights.
c_uno = concordance_index_ipcw(y, y, risk)[0]
print("Harrell C:", round(c_harrell, 3), " Uno C:", round(c_uno, 3))

# Time-dependent AUC and IBS need predicted SURVIVAL FUNCTIONS over a time grid.
times = np.percentile(time[event], np.linspace(10, 80, 8))
surv_fns = model.predict_survival_function(X)
surv_prob = np.row_stack([fn(times) for fn in surv_fns])   # S(t|x) per row

auc_t, auc_mean = cumulative_dynamic_auc(y, y, risk, times)
ibs = integrated_brier_score(y, y, surv_prob, times)
print("time-dependent AUC (mean):", round(auc_mean, 3), " IBS:", round(ibs, 3))

### Step 3 — A lifelines cross-check and anomaly-detection PR-AUCAs a sanity check we recompute the C-index with **lifelines**, which expects a predicted *time* rather than a risk, so we pass `-risk` (high risk = short time). We also fit a **Kaplan–Meier** population survival curve.The second family is **anomaly detection**: for rare events, accuracy is gamed by the majority class, so we use **PR-AUC** (average precision), which focuses on how well the rare positives are ranked.

In [ ]:
# SURVIVAL cross-check with lifelines (classic C-index + Kaplan-Meier curve).
from lifelines.utils import concordance_index
from lifelines import KaplanMeierFitter

# lifelines wants a PREDICTED TIME; pass -risk so that high risk = short time.
c_ll = concordance_index(time, -risk, event_observed=event)
km = KaplanMeierFitter().fit(time, event_observed=event)   # population survival curve

# ANOMALY DETECTION: PR-AUC is sklearn's average_precision_score.
from sklearn.metrics import average_precision_score, precision_recall_curve

# y_true: 1 = anomaly (rare); scores: higher = more anomalous.
y_true = np.array([0, 0, 0, 1, 0, 0, 1, 0])
scores = np.array([.1, .2, .3, .9, .15, .25, .7, .05])

pr_auc = average_precision_score(y_true, scores)           # PR-AUC / Average Precision
prec, rec, thr = precision_recall_curve(y_true, scores)
print("PR-AUC:", round(pr_auc, 3))

## Visualize the data & results

_On real load_diabetes data, treating the target as a time-to-event score, how well does each predictor's risk ranking match reality — i.e. what is the C-index for a real feature, a fitted model, and pure random?_

### Step 1 — Define the C-index from scratchTo make the concordance idea concrete, we write Harrell's C-index by hand. A pair of patients is **comparable** when one has a strictly earlier event time; it is **concordant** when that earlier-failing patient also got the higher risk score.We count concordant pairs (ties count as half) over all comparable pairs. We treat the diabetes target as an event 'time' so we have real data to score.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

d = load_diabetes()
X, y = d.data, d.target            # treat the target as an event 'time'
names = d.feature_names

def c_index(risk, time):
    """Fraction of comparable pairs ranked correctly (Harrell's C).
    Pair (i, j) is comparable when time[i] < time[j]; concordant when the
    earlier-event row got the higher risk. Ties count as half."""
    n = len(time)
    num = 0.0
    den = 0.0
    for i in range(n):
        for j in range(n):
            if time[i] < time[j]:          # i reaches the event first
                den += 1
                if risk[i] > risk[j]:
                    num += 1.0
                elif risk[i] == risk[j]:
                    num += 0.5
    return num / den

### Step 2 — Score three different risk rankingsWe compare three ways of ranking risk against the held-out target. First, a **single real feature** (BMI rises with progression, so as a risk score we use `-BMI`). Second, a **fitted linear model's** predicted time (again negated to make a risk). Third, a **pure random** score.The fitted model should rank best, the single feature a bit worse, and the random score should land near 0.5 — the no-skill baseline.

In [ ]:
# 1) A single REAL feature as a risk score. BMI rises with progression,
#    so as a 'risk' (high = sooner event) we use -BMI.
bmi_column = X[:, names.index("bmi")]
risk_feature = -bmi_column
c_feature = c_index(risk_feature, y)

# 2) A fitted linear model: predict the time, then risk = -predicted_time.
pred_time = LinearRegression().fit(X, y).predict(X)
c_model = c_index(-pred_time, y)

# 3) A random score -> should land near 0.5.
rng = np.random.default_rng(0)
random_scores = rng.standard_normal(len(y))
c_random = c_index(random_scores, y)

print(round(c_feature, 3), round(c_model, 3), round(c_random, 3))
# -> 0.695  0.755  0.504

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In a 3-year cancer study, patient P relapsed at 14 months and patient Q is still relapse-free when the study ends at 36 months. The model gave P a risk score of 0.7 and Q a risk score of 0.4. Is this pair comparable for the C-index, and is it concordant?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Classify each row's censoring. — _P had the event (relapse at 14 months), so P is uncensored. Q reached the end with no event, so Q is right-censored at 36 months: we only know Q's relapse time is > 36 months._
- Check comparability. — _P had the event at 14 months, which is before Q's observed time of 36 months, so we definitely know P relapsed first. The pair is comparable._
- Check concordance. — _The one who had the event first (P) should have the higher risk. P's risk 0.7 > Q's risk 0.4, so the model ranked them correctly._

**Answer:** Yes on both counts. Q is right-censored at 36 months (relapse time known only to be > 36 months), but because P relapsed at 14 months — before Q's 36-month observed time — we know P had the event first, so the pair is comparable. The model gave the earlier-failing patient (P) the higher risk (0.7 > 0.4), so the pair is concordant and counts as a correctly ranked pair in the C-index.

</details>

**Problem 2.** A spam-call detector is "99.5% accurate" on a stream where 0.5% of calls are scams. The security team is unimpressed. What is wrong with accuracy here, and which metrics should they ask for?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find the base rate. — _Only 0.5% of calls are scams, so 99.5% are legitimate. A model that flags nothing is correct on all 99.5% and scores 99.5% accuracy while catching zero scams._
- See that accuracy is gamed by imbalance. — _The 99.5% comes entirely from the easy majority class, not from any skill at finding the rare scam calls — the thing the team cares about._
- Pick metrics tied to the rare class. — _PR-AUC, and precision/recall at the team's alarm budget, measure how well the rare scams are actually caught; detection delay measures how fast._

**Answer:** The 99.5% is just the base rate of legitimate calls — a do-nothing detector that never alarms scores the same. Accuracy is meaningless on rare-event data. The team should ask for PR-AUC (Precision–Recall Area Under the Curve), and precision and recall at their alarm budget (how many of the top-$k$ flagged calls per day are real scams, and what fraction of scams that catches). For a live stream, also detection delay: once a scam campaign starts, how long until the first alarm.

</details>

**Problem 3.** Your survival model reports a Harrell's C-index of 0.52 and an IBS (Integrated Brier Score) of 0.24. A teammate says "C is fine, ship it." How do you read these two numbers?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Interpret the C-index scale. — _C = 1.0 is perfect risk ranking, C = 0.5 is random (a coin flip). 0.52 is barely above random — the model has essentially no useful ranking signal._
- Interpret the IBS scale. — _Lower IBS is better; 0 is perfect. A naive model that just predicts the overall survival curve lands around 0.25, so 0.24 is barely better than that baseline._
- Combine the verdict. — _Both the ranking metric and the calibration metric say the model is essentially at the no-skill baseline; nothing here justifies shipping._

**Answer:** Both numbers say the model has almost no skill. A C-index of 0.52 is barely above the random value of 0.50 — the risk ranking is essentially a coin flip. An IBS of 0.24 is right at the ~0.25 you get from a naive model that just predicts the population's average survival curve, so the predicted probabilities are no better calibrated than that baseline. The C is not fine; do not ship. Investigate features, censoring handling, and model fit before reporting again.

</details>